<a href="https://colab.research.google.com/github/ArlLps/Facom_LLMs/blob/main/GSI073_aula0_support_vector_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GSI073 - Tópicos Especiais de Inteligência Artificial

Neste notebook, um tipo de Support Vector Machine Linear.

## Preparação dos dados

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets
from sklearn.model_selection import train_test_split

# Preparar o dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

X = X[y != 1] ; y = y[y != 1] # remove versicolor
y = torch.tensor(y, dtype=torch.float32)
y[y == 0] = -1  # SVM espera rótulos em {-1, +1}

X = torch.tensor(X, dtype=torch.float32) # Tensor é um tipo especial que suporta muitas dimensões

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

A nossa Support Vector Machine é basicamente um hiperplano definido por *w* e *b* que melhor separa as classes.

In [ ]:
# Definir a Support Vector Machine como uma camada linear
n_features = X_train.shape[1]
model = nn.Linear(n_features, 1) # Equivalente a w e b

# === Hiperparâmetros ===
learning_rate = 0.01
epochs = 300
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Execução do treinamento

In [ ]:
for epoch in range(epochs):
    optimizer.zero_grad()

    y_pred_train = model(X_train)  # Modelo SVM (agora usando nn.Linear)

    # Hinge loss: max(0, 1 - y_i * (w^T x_i + b))
    perda_de_classificacao = torch.clamp(1 - y_train.view(-1, 1) * y_pred_train, min=0).mean()

    # Termo de regularização
    perda_de_distancia_entre_classes = 0.5 * torch.sum(model.weight ** 2) # 2/||w|| é a distância que queremos que seja a maior possível

    # Função objetivo tradicional: minimizar reg + C * hinge
    loss = perda_de_distancia_entre_classes + perda_de_classificacao

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss={loss.item():.4f}")

# Evaluate on test set
with torch.no_grad():
    y_pred_test = model(X_test)
    accuracy = ((y_test.view(-1, 1) * y_pred_test) > 0).float().mean()
    print(f"\nAccuracy on test set: {accuracy.item():.4f}")